# DOSE — YOLOv12n Training (250 epochs)

YOLOv12n (2.6M params) on 9,502 Vietnamese pill images (108 classes).

AdamW optimizer, mixup=0.2, copy_paste=0.1 for better augmentation.

In [ ]:
!pip install -q ultralytics

In [ ]:
from ultralytics import YOLO
import os, shutil, json, random, glob
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

DATASET_PATH = "/kaggle/input/dose-yolo-dataset"

# Load class names
with open(os.path.join(DATASET_PATH, "class_names.json")) as f:
    class_names_dict = json.load(f)
class_names_list = [class_names_dict[str(i)] for i in range(108)]
NC = len(class_names_list)
print(f"Loaded {NC} class names")

# Patch YAML path for Kaggle
yaml_path = os.path.join(DATASET_PATH, "dataset.yaml")
with open(yaml_path) as f:
    content = f.read()
content = content.replace("path: /home/lducc/code/dose/data/yolo", f"path: {DATASET_PATH}")
working_yaml = "/kaggle/working/dataset.yaml"
with open(working_yaml, "w") as f:
    f.write(content)
print("Dataset ready:", working_yaml)

# Verify dataset
train_count = len(list(Path(DATASET_PATH).glob("images/train/*.jpg")))
val_count = len(list(Path(DATASET_PATH).glob("images/val/*.jpg")))
print(f"Train: {train_count} images, Val: {val_count} images")

In [ ]:
# YOLOv12n — 250 epochs, AdamW, augmented
model = YOLO("yolo12n.pt")

results = model.train(
    data=working_yaml,
    epochs=250,
    imgsz=640,
    batch=32,
    patience=30,
    device=0,
    name="dose_yolo12n_250ep",
    exist_ok=True,
    optimizer="AdamW",
    lr0=0.001,
    lrf=0.01,
    weight_decay=0.001,
    cos_lr=True,
    mixup=0.2,
    copy_paste=0.1,
    mosaic=1.0,
    scale=0.5,
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    close_mosaic=15,
    warmup_epochs=3.0,
)

In [ ]:
model.export(format="onnx", imgsz=640, half=True)

In [ ]:
# Find and copy ONNX
onnx_src = glob.glob("/kaggle/working/runs/detect/dose_yolo12*/weights/best.onnx")
if onnx_src:
    shutil.copy(onnx_src[0], "/kaggle/working/vaipe.onnx")
    print(f"ONNX: {onnx_src[0]} -> /kaggle/working/vaipe.onnx")
    print(f"Size: {os.path.getsize('/kaggle/working/vaipe.onnx') / 1e6:.1f} MB")
else:
    print("ONNX not found! Check runs/")

In [ ]:
csv_files = glob.glob("/kaggle/working/runs/detect/dose_yolo12*/results.csv")
for f in csv_files:
    dst = os.path.join("/kaggle/working", os.path.basename(f))
    shutil.copy(f, dst)
    print(f"Copied: {dst}")

In [ ]:
# Benchmark: validation metrics
# Find best weights
best_weights = glob.glob("/kaggle/working/runs/detect/dose_yolo12*/weights/best.pt")
model = YOLO(best_weights[0])
print(f"Loading: {best_weights[0]}")

metrics = model.val(data=working_yaml, split="val")

print(f"\n{'='*50}")
print(f"BENCHMARK RESULTS")
print(f"{'='*50}")
print(f"mAP50:     {metrics.box.map50:.4f}")
print(f"mAP50-95:  {metrics.box.map:.4f}")
print(f"Precision: {metrics.box.mp:.4f}")
print(f"Recall:    {metrics.box.mr:.4f}")
print(f"{'='*50}")

# Per-class metrics — ap is always length nc
nc = 108

# metrics.box.ap = per-class mAP50 (always length nc)
# metrics.box.ap50 = same as ap for mAP50
ap50 = metrics.box.ap.tolist() if hasattr(metrics.box, "ap") else [0.0] * nc
ap50 = ap50[:nc]  # truncate if longer
while len(ap50) < nc:
    ap50.append(0.0)

# p and r may be shorter — safe-pad them
def safe_pad(arr, target_len):
    """Convert to list, handle nested arrays, pad to target length."""
    if arr is None or (hasattr(arr, '__len__') and len(arr) == 0):
        return [0.0] * target_len
    if hasattr(arr, 'tolist'):
        arr = arr.tolist()
    result = []
    for x in arr[:target_len]:
        if isinstance(x, (list, tuple, np.ndarray)):
            result.append(float(np.mean(x)))
        else:
            result.append(float(x))
    while len(result) < target_len:
        result.append(0.0)
    return result

p_vals = safe_pad(getattr(metrics.box, "p", None), nc)
r_vals = safe_pad(getattr(metrics.box, "r", None), nc)

df = pd.DataFrame({
    "class_id": range(nc),
    "name": class_names_list,
    "mAP50": ap50,
    "precision": p_vals,
    "recall": r_vals,
})
per_class_path = "/kaggle/working/per_class_metrics.csv"
df.to_csv(per_class_path, index=False)
print(f"\nSaved: {per_class_path}")

# Top 10 best and worst classes
df_sorted = df.sort_values("mAP50", ascending=False)
print(f"\nTop 10 classes (by mAP50):")
for _, row in df_sorted.head(10).iterrows():
    print(f"  {row['mAP50']:.3f}  {row['name']}")
print(f"\nBottom 10 classes (by mAP50):")
for _, row in df_sorted.tail(10).iterrows():
    print(f"  {row['mAP50']:.3f}  {row['name']}")


In [ ]:
# Energy-Based OOD (Out-of-Distribution) Detection
# Energy = -log(conf), Entropy = binary entropy of top class
# Based on pilot run: conf mean=0.78, energy mean=0.28, entropy mean=0.46

REJECT_CONF_THRESHOLD = 0.40    # max prob below this → reject
REJECT_ENERGY_THRESHOLD = 0.80  # energy above this → reject (low confidence)
REJECT_ENTROPY_THRESHOLD = 0.60 # entropy above this → reject (ambiguous)

def compute_energy(probs):
    logits = np.log(probs + 1e-10)
    return -np.log(np.sum(np.exp(logits)))

def compute_entropy(probs):
    return -np.sum(probs * np.log(probs + 1e-10))

def classify_detection(conf, pred_class, max_prob, energy, entropy):
    if pred_class == 107:
        return "reject", "out_of_prescription"
    if max_prob < REJECT_CONF_THRESHOLD:
        return "reject", f"low_confidence ({max_prob:.3f})"
    if energy > REJECT_ENERGY_THRESHOLD:
        return "reject", f"unknown_pill (energy={energy:.2f})"
    if entropy > REJECT_ENTROPY_THRESHOLD:
        return "reject", f"ambiguous (entropy={entropy:.2f})"
    return "accept", "ok"

print("OOD thresholds:")
print(f"  Confidence:  {REJECT_CONF_THRESHOLD}")
print(f"  Energy:      {REJECT_ENERGY_THRESHOLD}")
print(f"  Entropy:     {REJECT_ENTROPY_THRESHOLD}")
print("Adjust these and re-run this cell to tune rejection.")

In [ ]:
# OOD benchmark: test rejection on val set
val_images = sorted(Path(DATASET_PATH).glob("images/val/*.jpg"))

reject_stats = {"accept": 0, "reject": 0, "reject_reasons": {}}
ood_scores = []

for img_path in val_images[:200]:
    r = model(str(img_path), verbose=False)[0]
    if len(r.boxes) == 0:
        reject_stats["reject"] += 1
        reject_stats["reject_reasons"]["no_detections"] = \
            reject_stats["reject_reasons"].get("no_detections", 0) + 1
        continue
    
    confs = r.boxes.conf.cpu().numpy()
    classes = r.boxes.cls.cpu().numpy().astype(int)
    
    for conf, cls in zip(confs, classes):
        max_prob = float(conf)
        pred_class = int(cls)
        energy = -np.log(max_prob + 1e-10)
        entropy = -(max_prob * np.log(max_prob + 1e-10) + (1-max_prob) * np.log(1-max_prob + 1e-10))
        decision, reason = classify_detection(max_prob, pred_class, max_prob, energy, entropy)
        reject_stats[decision] += 1
        if decision == "reject":
            reject_stats["reject_reasons"][reason.split("(")[0].strip()] = \
                reject_stats["reject_reasons"].get(reason.split("(")[0].strip(), 0) + 1
        ood_scores.append({"energy": energy, "entropy": entropy, "max_prob": max_prob})

total = reject_stats["accept"] + reject_stats["reject"]
print(f"OOD Analysis (first 200 val images):")
print(f"  Accepted: {reject_stats['accept']}/{total} ({reject_stats['accept']/max(total,1)*100:.1f}%)")
print(f"  Rejected: {reject_stats['reject']}/{total} ({reject_stats['reject']/max(total,1)*100:.1f}%)")
if reject_stats["reject_reasons"]:
    print(f"  Rejection reasons:")
    for reason, count in sorted(reject_stats["reject_reasons"].items(), key=lambda x: -x[1]):
        print(f"    {reason}: {count}")
if ood_scores:
    energies = [s["energy"] for s in ood_scores]
    entropies = [s["entropy"] for s in ood_scores]
    probs_list = [s["max_prob"] for s in ood_scores]
    print(f"\nScore distributions (per-box):")
    print(f"  Energy:  mean={np.mean(energies):.2f}, std={np.std(energies):.2f}, min={np.min(energies):.2f}, max={np.max(energies):.2f}")
    print(f"  Entropy: mean={np.mean(entropies):.2f}, std={np.std(entropies):.2f}")
    print(f"  Conf:    mean={np.mean(probs_list):.3f}, std={np.std(probs_list):.3f}")
    print(f"\nSuggested thresholds (1 std below mean):")
    print(f"  REJECT_CONF_THRESHOLD  = {max(0.3, np.mean(probs_list) - np.std(probs_list)):.3f}")
    print(f"  REJECT_ENERGY_THRESHOLD = {np.mean(energies) + np.std(energies):.2f}")
    print(f"  REJECT_ENTROPY_THRESHOLD = {np.mean(entropies) + np.std(entropies):.2f}")

In [ ]:
# Full val inference: score every image (batched to avoid OOM)
val_dir = Path(DATASET_PATH) / "images" / "val"
val_images = sorted(val_dir.glob("*.jpg"))
print(f"Running inference on {len(val_images)} validation images...")

BATCH = 32
image_scores = []
for i in range(0, len(val_images), BATCH):
    batch = [str(p) for p in val_images[i:i+BATCH]]
    results_batch = model(batch, verbose=False)
    for img_path, r in zip(val_images[i:i+BATCH], results_batch):
        confs = r.boxes.conf.cpu().numpy() if len(r.boxes) > 0 else np.array([0.0])
        image_scores.append({
            "path": img_path,
            "avg_conf": float(np.mean(confs)),
            "n_boxes": len(r.boxes),
        })
    if (i // BATCH) % 10 == 0:
        print(f"  {min(i+BATCH, len(val_images))}/{len(val_images)} done")

# Select 16 random + 16 worst-performing (lowest avg conf)
random.seed(42)
random_16 = random.sample(image_scores, 16)
worst_16 = sorted(image_scores, key=lambda x: x["avg_conf"])[:16]

print(f"Random samples avg confidence: {np.mean([s['avg_conf'] for s in random_16]):.3f}")
print(f"Worst samples avg confidence: {np.mean([s['avg_conf'] for s in worst_16]):.3f}")

In [ ]:
def plot_grid(samples, title, save_path):
    fig, axes = plt.subplots(4, 4, figsize=(20, 20))
    fig.suptitle(title, fontsize=16, y=1.02)
    for ax, s in zip(axes.flatten(), samples):
        r = model(str(s["path"]), verbose=False)[0]
        plotted = r.plot(labels=True, conf=True)
        ax.imshow(cv2.cvtColor(plotted, cv2.COLOR_BGR2RGB))
        ax.set_title(f"{s['path'].name}\nconf={s['avg_conf']:.2f}, n={s['n_boxes']}", fontsize=8)
        ax.axis("off")
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved: {save_path}")

plot_grid(random_16, "16 Random Validation Samples", "/kaggle/working/inference_random.png")


In [ ]:
plot_grid(worst_16, "16 Worst-Performing Samples (Lowest Avg Confidence)", "/kaggle/working/inference_worst.png")
